In [1]:
import os, sys, time

from requests import options
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../')))
from src.utils import *
from src.panda_program import PandaMugProgram
from src.generic_program import ProgramOptions
from pydrake.all import (
    StartMeshcat,
)
from tqdm import tqdm


ikflow/config.py | Using device: 'cuda:0'


In [2]:

####### Options #######
num_tests = 100
num_initial_guesses = 10
program_options = ProgramOptions(
    visualize=True,
    joint_centering_cost=0,
    max_wall_time=60.0,
    which_solver='ipopt',
    acceptable_tol = 1e-3,
    acceptable_constr_viol_tol = 1e-4,
    ik_constraint_tol = (1e-6, 0.01),
    mug_height = 0.04,
    vars_file = "vars_file.txt"
)

In [3]:

meshcat = StartMeshcat()
mug_meshcat = StartMeshcat()

yaml_file = os.path.join(RepoDir(), "models/panda/panda_finray_collision.yaml")
base_diagram = BuildEnv(meshcat=meshcat, directives_file = yaml_file)
program = PandaMugProgram(base_diagram)
program.create_prog()


start = time.time()
i = 0
q = np.zeros(7)
qs = np.zeros((num_tests, 7))
targets = np.zeros((num_tests, 7))
while i < num_tests:
    q = np.random.uniform(program.plant.GetPositionLowerLimits(), program.plant.GetPositionUpperLimits())
    program.plant.SetPositions(program.plant_context, q)
    if program.collision_free_constraint_eval.Eval(q) < 1:
        pose = program.frame.CalcPoseInWorld(program.plant_context)
        targets[i] = np.array([*pose.translation(), *pose.rotation().ToQuaternion().wxyz()])
        qs[i] = q
        i += 1
print("Generated {} collision-free targets in {:.2f} seconds".format(num_tests, time.time() - start))

ik_solver = program.ik_solver

successes = 0
times = []
costs = []

INFO:drake:Meshcat listening for connections at http://localhost:7000
INFO:drake:Meshcat listening for connections at http://localhost:7001


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link6.dae into Group
Geometry: Loa

In [ ]:
i = 5
diagram_with_mug, mug = GenerateDiagramWithMug(qs[i], program, yaml_file, mug_meshcat)
with HiddenPrints():
    mug_program = PandaMugProgram(diagram_with_mug, options=program_options, model=ik_solver)
    mug_program.SetPositions(qs[i])
# for j in range(num_initial_guesses):
mug_program.create_prog(target_mug=mug)
mug_program.options.file_print_name = RepoDir() + f"/results/panda/mug/learned/collision_test_{i}.txt"
start = time.time()

result = mug_program.Solve()
if not result.is_success():
    print("Failed IK for target {} in {:.2f} seconds".format(i, time.time() - start))
else:
    print("Solved IK for target {} in {:.2f} seconds".format(i, time.time() - start))
    # print(result.get_optimal_cost() / mug_program.options.joint_centering_cost)
    times.append(time.time() - start)
    # costs.append(result.get_optimal_cost() / mug_program.options.joint_centering_cost)
    successes += 1
del mug_program

# print("Solved {} / {} targets in {:.2f} seconds".format(successes, num_tests, sum(times)))
# print("Average cost: {:.2f}".format(sum(costs) / len(costs) if costs else 0))
# print("Average time: {:.2f}".format(sum(times) / len(times) if times else 0))



Solved IK for target 5 in 3.07 seconds


ZeroDivisionError: float division by zero

In [15]:
mug_program.plant.GetPositions(mug_program.plant_context)

array([ 1.55082175, -0.58495735, -0.35950158, -2.11455154,  0.39571132,
        2.46577016,  0.43783152])

In [17]:
meshcat2 = StartMeshcat()
diagram2, _ = GenerateDiagramWithMug(qs[5], program = mug_program, yaml_file = os.path.join(RepoDir(), "models/panda/panda_finray_collision_backup.yaml"), meshcat = meshcat2)

INFO:drake:Meshcat listening for connections at http://localhost:7002


In [22]:
diagram_context2 = diagram2.CreateDefaultContext()
diagram2.ForcedPublish(diagram_context2)
plant2 = diagram2.GetSubsystemByName("plant")
plant_context2 = plant2.GetMyContextFromRoot(diagram_context2)

In [42]:
plant2.GetPositions(plant_context2)
three_sols = np.zeros(21)
three_sols[:7] = np.array([ 0.75653863, -0.27240363,  0.82476366, -2.04120159, -1.47375679, 1.47313178, -1.46885872])
three_sols[7:14] = np.array([ 0.8874414 , -0.59253049,  0.36857331, -2.1847043 ,  2.13744855, 3.13340878,  2.05150366])
three_sols[14:] = np.array([ 1.55082175, -0.58495735, -0.35950158, -2.11455154,  0.39571132, 2.46577016,  0.43783152])
plant2.SetPositions(plant_context2, three_sols)

opacity = 1
meshcat2.SetProperty(f"/drake/illustration/panda", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/finray", "opacity", 0.7)
meshcat2.SetProperty(f"/drake/illustration/panda2", "opacity", 0)
meshcat2.SetProperty(f"/drake/illustration/finray2", "opacity", 0)
meshcat2.SetProperty(f"/drake/illustration/panda3", "opacity", 0)
meshcat2.SetProperty(f"/drake/illustration/finray3", "opacity", 0)
diagram2.ForcedPublish(diagram_context2)